# THINGS ResNet-50 Baseline on Colab

Open this notebook in VS Code and select a Google Colab GPU kernel. The notebook calls the project scripts; it does not duplicate training logic.

Training should run from `/content/human-things`, not directly from Google Drive, because THINGS has many small image files.

In [ ]:
import sys
from pathlib import Path

try:
    import torch
    print('Python:', sys.version)
    print('Torch:', torch.__version__)
    print('CUDA:', torch.cuda.is_available())
    print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
except Exception as exc:
    print('Torch check failed:', repr(exc))

## Configure

`REPO_URL` is set to the project GitHub repo. `DRIVE_DATA_ROOT` should contain a `data/` folder with `raw/` and `processed/` inside it. Change `DRIVE_DATA_ROOT` only if your Google Drive folder has a different name.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/sabszh/human-things.git'
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/human-things-data')
PROJECT_ROOT = Path('/content/human-things')

print('Drive data root:', DRIVE_DATA_ROOT)
print('Project root:', PROJECT_ROOT)

## Clone or Update Repo

In [ ]:
if not PROJECT_ROOT.exists():
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    %cd {PROJECT_ROOT}
    !git pull

%cd {PROJECT_ROOT}
!find scripts -maxdepth 1 -type f -printf '%f\n' | sort

## Copy Data to Colab Local Disk

This copies Drive data into `/content/human-things/data`. It can take a while, but it avoids slow training from Drive.

In [ ]:
source_data = DRIVE_DATA_ROOT / 'data'
target_data = PROJECT_ROOT / 'data'

if not source_data.exists():
    raise FileNotFoundError(f'Drive data folder not found: {source_data}')

!mkdir -p {target_data}
!rsync -a --info=progress2 {source_data}/ {target_data}/

%cd {PROJECT_ROOT}
!find data -maxdepth 3 -type f -printf '%p %s\n' | sort | head -60
!echo THINGS image count:
!find data/raw/THINGS-database/osfstorage/images_THINGS/object_images -type f -name '*.jpg' | wc -l
!echo THINGSplus CC0 image count:
!find data/raw/THINGS-database/osfstorage/images_THINGSplus-CC0/object_images_CC0 -maxdepth 1 -type f -name '*.jpg' | wc -l

## Install Requirements and Check GPU

In [ ]:
%cd {PROJECT_ROOT}
!python -m pip install -q -r requirements.txt

import torch, torchvision
print('torch', torch.__version__)
print('torchvision', torchvision.__version__)
print('cuda', torch.cuda.is_available())
print('device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Rebuild Metadata and Splits

In [ ]:
%cd {PROJECT_ROOT}
!python scripts/01_make_metadata_csv.py
!python scripts/02_make_image_splits.py
!cat outputs/image_metadata_report.json
!cat outputs/image_splits_report.json

## Dry Run Data Loading

In [ ]:
%cd {PROJECT_ROOT}
!python scripts/03_train_resnet50_image_only.py --dry-run --batch-size 16 --num-workers 2

## Train

Start with the short run. Only set `RUN_FULL = True` after the short run finishes cleanly.

In [ ]:
%cd {PROJECT_ROOT}

RUN_SHORT = True
RUN_FULL = False
BATCH_SIZE = 64
NUM_WORKERS = 2

if RUN_SHORT:
    !python scripts/03_train_resnet50_image_only.py --head-epochs 1 --layer4-epochs 1 --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS}

if RUN_FULL:
    !python scripts/03_train_resnet50_image_only.py --head-epochs 5 --layer4-epochs 10 --batch-size {BATCH_SIZE} --num-workers {NUM_WORKERS}

## Inspect Training Outputs

In [ ]:
%cd {PROJECT_ROOT}
!find outputs/baseline_resnet50 -maxdepth 2 -type f -printf '%p %s\n' | sort || true
!test -f outputs/baseline_resnet50/metrics.json && cat outputs/baseline_resnet50/metrics.json || true
!test -f outputs/baseline_resnet50/training_log.csv && head -20 outputs/baseline_resnet50/training_log.csv || true

## Extract and Evaluate Embeddings

Run after `outputs/baseline_resnet50/best_model.pt` exists.

In [ ]:
%cd {PROJECT_ROOT}

RUN_EXTRACT_AND_EVAL = False

if RUN_EXTRACT_AND_EVAL:
    !python scripts/04_extract_resnet50_embeddings.py --batch-size 128 --num-workers 2
    !python scripts/05_evaluate_resnet50_embeddings.py
    !cat outputs/baseline_resnet50/embedding_eval_report.json

## Copy Results Back to Drive

In [ ]:
RUN_COPY_BACK = False
DRIVE_OUTPUT_ROOT = DRIVE_DATA_ROOT / 'outputs'

if RUN_COPY_BACK:
    !mkdir -p {DRIVE_OUTPUT_ROOT}
    !rsync -a --info=progress2 outputs/baseline_resnet50/ {DRIVE_OUTPUT_ROOT}/baseline_resnet50/
    !find {DRIVE_OUTPUT_ROOT}/baseline_resnet50 -maxdepth 3 -type f -printf '%p %s\n' | sort